# 01 — MarineSee dataset inventory

Count what is on Google Drive. **Do not train.** The study allows **nine** Mati species only.

Drive folder: [Seagrass](https://drive.google.com/drive/folders/1MU31a5BG3MKn1fEYV-ljhc2zS4CoE79F) (`1MU31a5BG3MKn1fEYV-ljhc2zS4CoE79F`).

Stop after this notebook if:
- a study species is missing
- `Halophila spinulosa` (or any other extra taxon) would be mixed into training
- `research-grade` vs `all` counts look wrong
- a `- Copy` folder is the only copy of a class

Classifier later is **SVM**, not YOLO.

## A. Mount Drive

Colab can only mount **the same Google account** that is signed into this notebook. The dataset can stay Shared-with-me if you already added a **My Drive** shortcut named `Seagrass`.

If `drive.mount` says `credential propagation was unsuccessful`:

1. **Runtime → Disconnect and delete runtime**.
2. Stay on **one** Google account (the one that has the `Seagrass` shortcut). Do not pick akeroshii in the popup.
3. When Google asks for access, allow **all** Drive permissions (partial allow fails).
4. Or skip the code: left sidebar **Files** (folder icon) → **Mount Drive**, wait until `drive/` appears, then run the cells again.

In [ ]:
from pathlib import Path
import json
import sys

IN_COLAB = Path("/content").exists()
print("colab", IN_COLAB)

DRIVE_MOUNTED = Path("/content/drive/MyDrive").exists() or Path("/content/drive/My Drive").exists()
print("drive already mounted", DRIVE_MOUNTED)

if IN_COLAB and not DRIVE_MOUNTED:
    from google.colab import drive

    try:
        drive.mount("/content/drive", force_remount=True)
    except Exception as exc:
        print(type(exc).__name__, exc)
        print(
            "Mount failed. Use ONE Google account (the one with the Seagrass shortcut). "
            "In the popup, allow ALL Drive permissions. "
            "Or: left sidebar Files → Mount Drive, then re-run this cell."
        )
        raise

print("mount ok" if (Path("/content/drive/MyDrive").exists() or Path("/content/drive/My Drive").exists()) else "mount missing")

: 

In [ ]:
# Change this if your shortcut is not named Seagrass.
CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/Seagrass"),
    Path("/content/drive/My Drive/Seagrass"),
]

MARKER = "cymodocea rotundata"


def looks_like_dataset_root(path: Path) -> bool:
    if not path.is_dir():
        return False
    names = " ".join(p.name.lower() for p in path.iterdir() if p.is_dir())
    return MARKER in names


def find_root() -> Path:
    for cand in CANDIDATE_ROOTS:
        if looks_like_dataset_root(cand):
            return cand
    search_bases = [Path("/content/drive/MyDrive"), Path("/content/drive/My Drive")]
    for base in search_bases:
        if not base.is_dir():
            continue
        for path in base.rglob("*"):
            if path.is_dir() and looks_like_dataset_root(path):
                return path
    raise FileNotFoundError(
        "Could not find the Seagrass dataset under My Drive. "
        "Add a shortcut to My Drive, or set DATA_ROOT manually in the next cell."
    )

DATA_ROOT = find_root()
print("DATA_ROOT", DATA_ROOT)

## B. Study contract

Nine classes only. Drive titles do not use our slugs; we match on the scientific name inside the folder title.

In [ ]:
REPO_SRC = Path("/content/seagrass/ml/src")
if not REPO_SRC.is_dir():
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / "ml" / "src"
        if (candidate / "dataset_contract.py").is_file():
            REPO_SRC = candidate
            break

if (REPO_SRC / "dataset_contract.py").is_file():
    sys.path.insert(0, str(REPO_SRC.parent))
    from src.dataset_contract import (
        IMAGE_SUFFIXES,
        STUDY_SPECIES,
        classify_folder,
    )
    from src.inventory import scan_root
    print("imported ml/src from", REPO_SRC)
else:
    print("ml/src not on this machine — using inlined contract")
    STUDY_SPECIES = (
        "cymodocea_rotundata",
        "cymodocea_serrulata",
        "enhalus_acoroides",
        "halodule_pinifolia",
        "halodule_uninervis",
        "halophila_minor",
        "halophila_ovalis",
        "syringodium_isoetifolium",
        "thalassia_hemprichii",
    )
    SPECIES_PHRASES = {
        "cymodocea_rotundata": "cymodocea rotundata",
        "cymodocea_serrulata": "cymodocea serrulata",
        "enhalus_acoroides": "enhalus acoroides",
        "halodule_pinifolia": "halodule pinifolia",
        "halodule_uninervis": "halodule uninervis",
        "halophila_minor": "halophila minor",
        "halophila_ovalis": "halophila ovalis",
        "syringodium_isoetifolium": "syringodium isoetifolium",
        "thalassia_hemprichii": "thalassia hemprichii",
    }
    OUT_OF_SCOPE_PHRASES = {"halophila_spinulosa": "halophila spinulosa"}
    IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff")

    def _norm(name: str) -> str:
        return " ".join(name.lower().replace("_", " ").split())

    def classify_folder(name: str):
        n = _norm(name)
        variant = "unknown"
        if "copy" in n:
            variant = "copy"
        elif "research-grade" in n or "research grade" in n:
            variant = "research-grade"
        elif "dataset (all)" in n or n.endswith("(all)") or " (all)" in n:
            variant = "all"
        elif "dataset" in n:
            variant = "all"
        for slug, phrase in SPECIES_PHRASES.items():
            if phrase in n:
                return slug, "study", variant
        for slug, phrase in OUT_OF_SCOPE_PHRASES.items():
            if phrase in n:
                return slug, "extra", variant
        return None, "unknown", variant

    def _is_image(path: Path) -> bool:
        return path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES

    def _count_images(folder: Path) -> int:
        return sum(1 for p in folder.rglob("*") if _is_image(p))

    def scan_root(root: Path):
        from collections import defaultdict
        rows = []
        for child in sorted(root.iterdir(), key=lambda p: p.name.lower()):
            if not child.is_dir() or child.name.startswith("."):
                continue
            slug, scope, variant = classify_folder(child.name)
            rows.append({
                "folder": child.name,
                "path": str(child),
                "slug": slug,
                "scope": scope,
                "variant": variant,
                "images": _count_images(child),
            })
        by_study = {slug: defaultdict(int) for slug in STUDY_SPECIES}
        extra, unknown = [], []
        for row in rows:
            if row["scope"] == "study" and row["slug"]:
                by_study[row["slug"]][row["variant"]] += row["images"]
            elif row["scope"] == "extra":
                extra.append(row)
            else:
                unknown.append(row)
        missing = [s for s in STUDY_SPECIES if sum(by_study[s].values()) == 0]
        return {
            "root": str(root),
            "folders": rows,
            "by_study": {k: dict(v) for k, v in by_study.items()},
            "extra": extra,
            "unknown": unknown,
            "missing": missing,
        }

print("classes", len(STUDY_SPECIES))

## C. Count every folder

This can take a few minutes the first time Drive metadata is cold.

In [ ]:
report = scan_root(DATA_ROOT)

print("root", report["root"])
print()
print(f"{'scope':<8} {'variant':<16} {'images':>7}  folder")
print("-" * 88)
for row in report["folders"]:
    print(
        f"{row['scope']:<8} {row['variant']:<16} {row['images']:>7}  {row['folder']}"
    )

In [ ]:
print("Per study species (do not mix extra taxa into these rows)\n")
print(f"{'slug':<28} {'research-grade':>15} {'all':>8} {'copy':>8} {'other':>8}")
print("-" * 72)
for slug in STUDY_SPECIES:
    counts = report["by_study"][slug]
    rg = counts.get("research-grade", 0)
    all_ = counts.get("all", 0)
    copy = counts.get("copy", 0)
    other = sum(v for k, v in counts.items() if k not in {"research-grade", "all", "copy"})
    print(f"{slug:<28} {rg:>15} {all_:>8} {copy:>8} {other:>8}")

print()
if report["missing"]:
    print("MISSING from Drive:", ", ".join(report["missing"]))
else:
    print("All nine study species have at least one image folder.")

if report["extra"]:
    print("\nOUT OF STUDY — do not train on these:")
    for row in report["extra"]:
        print(f"  {row['folder']}  ({row['images']} images)")

if report["unknown"]:
    print("\nUNRECOGNIZED folders:")
    for row in report["unknown"]:
        print(f"  {row['folder']}  ({row['images']} images)")

## D. Sample images (study species only)

Prefer `research-grade` for the preview. Skip extra taxa.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image


def pick_folder(slug: str):
    study = [
        r for r in report["folders"]
        if r["slug"] == slug and r["scope"] == "study" and r["images"] > 0
    ]
    for preferred in ("research-grade", "all"):
        for r in study:
            if r["variant"] == preferred:
                return Path(r["path"]), r["variant"]
    return (Path(study[0]["path"]), study[0]["variant"]) if study else (None, None)


def first_image(folder: Path):
    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES:
            return p
    return None

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, slug in zip(axes.ravel(), STUDY_SPECIES):
    folder, variant = pick_folder(slug)
    ax.set_xticks([])
    ax.set_yticks([])
    if folder is None:
        ax.set_title(f"{slug}\nMISSING", fontsize=9)
        continue
    img_path = first_image(folder)
    if img_path is None:
        ax.set_title(f"{slug}\nno file", fontsize=9)
        continue
    ax.imshow(Image.open(img_path))
    ax.set_title(f"{slug}\n{variant}", fontsize=8)
plt.tight_layout()
plt.show()

## E. Save a small report

Images stay on Drive. This JSON is only counts. Download it and drop it in the repo as `ml/artifacts/inventory.json` after you are happy with the numbers.

In [ ]:
out_local = Path("/content/inventory.json")
out_drive = Path("/content/drive/MyDrive/Seagrass-ml-artifacts")
payload = {
    "drive_folder_id": "1MU31a5BG3MKn1fEYV-ljhc2zS4CoE79F",
    **report,
}
out_local.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("wrote", out_local)

if Path("/content/drive/MyDrive").is_dir():
    out_drive.mkdir(parents=True, exist_ok=True)
    dest = out_drive / "inventory.json"
    dest.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("wrote", dest)

## Next (only after the table looks right)

1. Decide **research-grade** vs **all** (recommendation: train v1 on research-grade; keep `all` for a later stress test).
2. Ignore **Halophila spinulosa** and any `- Copy` folder.
3. Then `02_preprocess_and_features.ipynb` (not written until inventory is accepted).

Do not open a YOLO trainer for MarineSee. The paper requires SVM.